In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/03.silver-helpers

In [0]:
from pyspark.sql import functions as F

In [0]:
bronze_table = f'{catalog_name}.{bronze_schema}.races'
silver_table = f'{catalog_name}.{silver_schema}.races'

**Step 1 - Read bronze races table**

In [0]:
races_df = (
    spark.read.table(bronze_table).filter(F.col("batch_id") == v_batch_id)
)

In [0]:
display(races_df)

**Step 2 - Keep only the columns required (Drop url column)**

In [0]:
races_selected_df = races_df.select("season",
                   "round",
                   "raceName",
                   "date",
                   "circuitId",
                   "ingestion_timestamp",
                   "source_file",
                   "batch_id"
                   )

**Step 3 & 4 - Standardise Column Names**

In [0]:
races_renamed_df = (
    races_selected_df
        .withColumnsRenamed({"circuitId":"circuit_id",
                            "raceName":"race_name",
                            "date":"race_date"
        })
)

**Step 5 - Remove duplicate records**

In [0]:
races_distinct_df = races_renamed_df.dropDuplicates(["season", "round"])

In [0]:
display(races_distinct_df)

**Step 6 - Transform values of columns `race_name` to Title Case**

In [0]:
races_final_df = (
    races_distinct_df
        .withColumn("race_name", F.initcap(F.col("race_name")))
)

**Step 7 - Write the transformed data to silver `races` table**

In [0]:
write_to_silver(
    input_df =  races_final_df,
    target_table = silver_table,
    merge_condition = "t.season = s.season AND t.round = s.round",
    columns_to_update = [
        "race_name",
        "race_date",
        "circuit_id",
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    ]
)

In [0]:
%sql
select * from formula1_incr.silver.races